# 01 — Download Allen Brain Observatory data

Identifies the 17 Layer 2/3 excitatory VISp containers, downloads their NWB files via AllenSDK (~22 GB), and verifies the sparse-noise stimulus is present in every Session C experiment.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

CACHE_DIR  = REPO_ROOT / 'data' / 'cache'
LISTS_DIR  = REPO_ROOT / 'data' / 'experiment_lists'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from allensdk.core.brain_observatory_cache import BrainObservatoryCache
boc = BrainObservatoryCache(manifest_file=str(CACHE_DIR / 'manifest.json'))

In [ ]:
all_experiments = boc.get_ophys_experiments()
df_all = pd.DataFrame(all_experiments)

df_visp = df_all[df_all['targeted_structure'] == 'VISp'].copy()

In [ ]:
df_three = df_visp[df_visp['session_type'].str.contains('three_session', na=False)].copy()

In [ ]:
containers = []
for container_id, group in df_three.groupby('experiment_container_id'):
    sessions = set(group['session_type'])
    has_A = any('A' in s for s in sessions)
    has_B = any('B' in s for s in sessions)
    has_C = any('C' in s for s in sessions)
    
    if has_A and has_B and has_C:
        containers.append({
            'container_id': container_id,
            'n_sessions': len(group),
            'sessions': list(sessions),
            'experiment_ids': group['id'].tolist(),
            'cre_line': group['cre_line'].iloc[0],
            'imaging_depth': group['imaging_depth'].iloc[0]
        })

df_containers = pd.DataFrame(containers)

In [ ]:
target_containers = df_containers[
    (df_containers['imaging_depth'] >= 150) &
    (df_containers['imaging_depth'] <= 250) &
    (df_containers['cre_line'].isin(['Slc17a7-IRES2-Cre', 'Cux2-CreERT2']))
].copy()

sample = target_containers.iloc[0]

In [ ]:
n_containers = len(target_containers)
batch_name = 'all_l23_excitatory'

download_containers = target_containers.sample(
    n=min(n_containers, len(target_containers)),
    random_state=42
)

batch_exp_ids = []
for _, container in download_containers.iterrows():
    batch_exp_ids.extend(container['experiment_ids'])

download_batch = df_three[df_three['id'].isin(batch_exp_ids)].copy()

batch_file = os.path.join(experiment_lists_dir, f'{batch_name}_experiments.csv')
container_file = os.path.join(experiment_lists_dir, f'{batch_name}_containers.csv')

download_batch.to_csv(batch_file, index=False)
download_containers.to_csv(container_file, index=False)

In [ ]:
BATCH_FILE = 'all_l23_excitatory_experiments.csv'

batch_path = os.path.join(experiment_lists_dir, BATCH_FILE)

if not os.path.exists(batch_path):
    raise FileNotFoundError("Run 01_explore_allen_2p.ipynb first")

batch_df = pd.read_csv(batch_path)
experiment_ids = batch_df['id'].tolist()

In [ ]:
downloaded_dir = os.path.join(cache_dir, 'ophys_experiment_data')

already_downloaded = []
if os.path.exists(downloaded_dir):
    for fname in os.listdir(downloaded_dir):
        if fname.endswith('.nwb'):
            exp_id = int(fname.replace('ophys_experiment_', '').replace('.nwb', ''))
            already_downloaded.append(exp_id)

to_download = [eid for eid in experiment_ids if eid not in already_downloaded]

if len(to_download) == 0:
    pass


In [ ]:
def download_with_retry(exp_id, max_retries=3):
    """Download experiment with retry logic"""
    for attempt in range(max_retries):
        try:
            dataset = boc.get_ophys_experiment_data(exp_id)
            return dataset, True
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 30 * (attempt + 1)
                time.sleep(wait)
            else:
                return None, False
    return None, False

In [ ]:
if len(to_download) > 0:
    
    start_time = time.time()
    success_count = 0
    failed_ids = []
    
    for i, exp_id in enumerate(to_download, 1):
        exp_info = batch_df[batch_df['id'] == exp_id].iloc[0]
        
        dataset, success = download_with_retry(exp_id)
        
        if success:
            success_count += 1
            
            elapsed = time.time() - start_time
            rate = success_count / elapsed * 3600
            remaining = len(to_download) - i
            eta = remaining / rate if rate > 0 else 0
            
        else:
            failed_ids.append(exp_id)
    
    if failed_ids:
        pass


In [ ]:
verified = []
for exp_id in experiment_ids:
    nwb_file = os.path.join(downloaded_dir, f'{exp_id}.nwb')
    exists = os.path.exists(nwb_file)
    size_mb = os.path.getsize(nwb_file) / (1024**2) if exists else 0
    
    verified.append({
        'experiment_id': exp_id,
        'exists': exists,
        'size_mb': size_mb
    })

verify_df = pd.DataFrame(verified)
total_size = verify_df['size_mb'].sum() / 1024

missing = verify_df[~verify_df['exists']]
if len(missing) > 0:
    pass

verify_path = os.path.join(experiment_lists_dir, f'downloaded_{BATCH_FILE}')
verify_df.to_csv(verify_path, index=False)

In [ ]:
batch_df['downloaded'] = batch_df['id'].isin(verify_df[verify_df['exists']]['experiment_id'])

if 'experiment_container_id' in batch_df.columns:
    containers = batch_df[batch_df['downloaded']].groupby('experiment_container_id')
    complete = sum(1 for _, g in containers if len(g) == 3)

In [ ]:
BATCH_FILE = 'all_l23_excitatory_experiments.csv'

batch_df = pd.read_csv(os.path.join(experiment_lists_dir, BATCH_FILE))
verify_df = pd.read_csv(os.path.join(experiment_lists_dir, f'downloaded_{BATCH_FILE}'))

downloaded = verify_df[verify_df['exists']]['experiment_id'].tolist()

In [ ]:
session_types = batch_df['session_type'].unique()
results = {}

for session_type in session_types:
    exp_ids = batch_df[
        (batch_df['session_type'] == session_type) & 
        (batch_df['id'].isin(downloaded))
    ]['id'].tolist()
    
    if len(exp_ids) == 0:
        continue
    
    exp_id = exp_ids[0]
    
    try:
        dataset = boc.get_ophys_experiment_data(exp_id)
        
        cell_ids = dataset.get_cell_specimen_ids()
        timestamps, dff = dataset.get_dff_traces()
        
        stimuli = dataset.list_stimuli()
        
        stim_details = []
        for stim in stimuli:
            try:
                stim_table = dataset.get_stimulus_table(stim)
                stim_details.append(stim)
            except:
                pass
        
        results[session_type] = {
            'n_neurons': len(cell_ids),
            'duration': timestamps[-1],
            'stimuli': stim_details
        }
        
    except Exception as e:
        results[session_type] = {'error': str(e)[:200]}

In [ ]:
expected = {
    'Session A': ['drifting_gratings', 'natural_movie'],
    'Session B': ['static_gratings', 'natural_scenes', 'natural_movie'],
    'Session C': ['locally_sparse_noise']
}

for session_type, data in results.items():
    if 'error' in data:
        continue
    
    session_letter = 'A' if 'A' in session_type else 'B' if 'B' in session_type else 'C'
    expected_stim = expected[f'Session {session_letter}']
    actual_stim = data['stimuli']
    
    for exp_stim in expected_stim:
        found = any(exp_stim in s for s in actual_stim)
        status = "✓" if found else "✗"
    
    if session_letter == 'C':
        sparse = [s for s in actual_stim if 'sparse_noise' in s]
        if sparse:
            pass


In [ ]:
batch_df['downloaded'] = batch_df['id'].isin(downloaded)
containers = batch_df[batch_df['downloaded']].groupby('experiment_container_id')

complete_containers = []
incomplete_containers = []

for container_id, group in containers:
    sessions = set(group['session_type'])
    has_A = any('A' in s for s in sessions)
    has_B = any('B' in s for s in sessions)
    has_C = any('C' in s for s in sessions)
    
    if has_A and has_B and has_C:
        complete_containers.append(container_id)
    else:
        incomplete_containers.append({
            'container_id': container_id,
            'sessions': list(sessions),
            'missing': [x for x in ['A', 'B', 'C'] if not locals()[f'has_{x}']]
        })

if incomplete_containers:
    for item in incomplete_containers[:5]:
        pass


In [ ]:
session_c = batch_df[
    (batch_df['session_type'].str.contains('C')) &
    (batch_df['id'].isin(downloaded))
]['id'].iloc[0]

dataset = boc.get_ophys_experiment_data(session_c)
cell_ids = dataset.get_cell_specimen_ids()

stimuli = dataset.list_stimuli()
sparse_stim = [s for s in stimuli if 'sparse_noise' in s.lower()][0]

stim_table = dataset.get_stimulus_table(sparse_stim)
timestamps, dff = dataset.get_dff_traces(cell_specimen_ids=[cell_ids[0]])

In [ ]:
container_neuron_counts = []

for container_id, group in batch_df[batch_df['downloaded']].groupby('experiment_container_id'):
    sessions = set(group['session_type'])
    has_A = any('A' in s for s in sessions)
    has_B = any('B' in s for s in sessions)
    has_C = any('C' in s for s in sessions)
    
    if not (has_A and has_B and has_C):
        continue
    
    exp_id = group['id'].iloc[0]
    try:
        dataset = boc.get_ophys_experiment_data(exp_id)
        n_neurons = len(dataset.get_cell_specimen_ids())
        
        container_neuron_counts.append({
            'container_id': container_id,
            'n_neurons': n_neurons,
            'cre_line': group['cre_line'].iloc[0],
            'imaging_depth': group['imaging_depth'].iloc[0],
            'experiment_ids': group['id'].tolist()
        })
        
    except Exception as e:
        pass

df_neuron_counts = pd.DataFrame(container_neuron_counts)
total_neurons = df_neuron_counts['n_neurons'].sum()
mean_neurons = df_neuron_counts['n_neurons'].mean()
std_neurons = df_neuron_counts['n_neurons'].std()

for cre in df_neuron_counts['cre_line'].unique():
    n = df_neuron_counts[df_neuron_counts['cre_line'] == cre]['n_neurons'].sum()
    containers = len(df_neuron_counts[df_neuron_counts['cre_line'] == cre])

neuron_count_file = os.path.join(experiment_lists_dir, 'container_neuron_counts.csv')
df_neuron_counts.to_csv(neuron_count_file, index=False)